<a href="https://colab.research.google.com/github/KanagalaSriHarshitha1515/RAG_Q-A/blob/main/RAG_Q_A_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG) Pipeline for Question Answering

**Stack:** Python, Hugging Face Transformers, Sentence-Transformers, LangChain, ChromaDB

**Pipeline:**

SQuAD → Data Cleaning → Duplicate Context Removal → LangChain Documents → LangChain Chunking → BM25 + ChromaDB Dense Retrieval → Reciprocal Rank Fusion (RRF) → Candidate Top-50 → Cross-Encoder Reranking → Top-3 → TinyLlama-1.1B-Chat → Answer → Precision / Recall / F1 / BERTScore

### Important evaluation design

This notebook keeps **retrieval corpus and evaluation questions separated conceptually**:

- A small **training-context development subset** is used only to choose the chunk size/overlap.
- The final RAG index is then rebuilt from **SQuAD validation contexts**.
- Validation questions are used as the final QA evaluation queries.
- Therefore the final retrieval test asks: *Can the retriever find the correct passage/chunk in the supplied validation knowledge base?*
- The notebook does **not** mix training contexts into the final validation index by accident.

SQuAD 1.1 officially contains 87,599 training examples and 10,570 validation examples. citeturn0search1turn0search2


## 1. Install dependencies

Run this cell first. If Colab asks for a runtime restart after package installation, restart the runtime and then use **Run all** again.

In [ ]:
!pip -q install -U datasets rank_bm25 sentence-transformers transformers accelerate
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb
!pip -q install -U bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 48.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/1

## 2. Imports and reproducibility

In [ ]:
import os
import re
import gc
import random
import shutil
import hashlib
import string
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch:", torch.__version__)

Device: cpu
Torch: 2.11.0+cpu


## 3. Load SQuAD 1.1

In [ ]:
dataset = load_dataset("rajpurkar/squad")

print(dataset)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

print("\nExample:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
Train examples: 87599
Validation examples: 10570

Example:
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a di

## 4. Data cleaning

The original context text is kept intact as much as possible because SQuAD contains answer offsets. We clean question/answer strings and remove invalid examples, but we do not aggressively rewrite context text before indexing.

In [ ]:
def clean_string(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def prepare_split(split):
    cleaned = []

    for row in split:
        context = str(row["context"]).strip()
        question = clean_string(row["question"])

        answer_texts = [
            clean_string(x)
            for x in row["answers"]["text"]
            if clean_string(x)
        ]

        answer_starts = list(row["answers"]["answer_start"])

        if not context or not question or not answer_texts:
            continue

        cleaned.append({
            "id": row["id"],
            "title": row["title"],
            "context": context,
            "question": question,
            "answers": answer_texts,
            "answer_starts": answer_starts
        })

    return cleaned

train_examples = prepare_split(dataset["train"])
validation_examples_all = prepare_split(dataset["validation"])

print("Clean train examples:", len(train_examples))
print("Clean validation examples:", len(validation_examples_all))

Clean train examples: 87599
Clean validation examples: 10570


## 5. Remove duplicate contexts while preserving the first original context

In [ ]:
def context_key(text):
    # Used only to detect duplicate/whitespace-equivalent contexts.
    return re.sub(r"\s+", " ", text).strip().lower()

def unique_context_records(examples):
    seen = {}
    records = []

    for row in examples:
        key = context_key(row["context"])

        if key not in seen:
            context_id = hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]
            seen[key] = context_id
            records.append({
                "context_id": context_id,
                "title": row["title"],
                "context": row["context"]
            })

    return records

train_context_records = unique_context_records(train_examples)
validation_context_records = unique_context_records(validation_examples_all)

print("Unique train contexts:", len(train_context_records))
print("Unique validation contexts:", len(validation_context_records))

Unique train contexts: 18891
Unique validation contexts: 2067


## 6. Development set for chunk-size tuning

Chunking is tuned **only on a development subset of training examples**. The final validation corpus is not used to choose the chunk size.

This avoids accidentally selecting the chunk configuration using the final evaluation questions.

In [ ]:
DEV_TUNING_N = min(100, len(train_examples))
dev_rng = random.Random(SEED)
dev_tuning_examples = dev_rng.sample(train_examples, DEV_TUNING_N)

print("Chunk-tuning development questions:", len(dev_tuning_examples))

Chunk-tuning development questions: 100


## 7. LangChain Documents

In [ ]:
def records_to_documents(records):
    docs = []

    for record in records:
        docs.append(
            Document(
                page_content=record["context"],
                metadata={
                    "context_id": record["context_id"],
                    "title": record["title"]
                }
            )
        )

    return docs

train_documents = records_to_documents(train_context_records)
validation_documents = records_to_documents(validation_context_records)

print("Train LangChain Documents:", len(train_documents))
print("Validation LangChain Documents:", len(validation_documents))
print(train_documents[0].metadata)

Train LangChain Documents: 18891
Validation LangChain Documents: 2067
{'context_id': '17b6b1553a46428b', 'title': 'University_of_Notre_Dame'}


## 8. LangChain chunking + chunk-size tuning

We compare several practical chunk configurations on the training development subset using BM25 retrieval.

The selected configuration is **not claimed to be globally optimal**. It is simply the configuration selected by this reproducible development experiment.

In [ ]:
CHUNK_CONFIGS = [
    (800, 100),
    (1200, 200),
    (1600, 300),
]

def make_splitter(chunk_size, chunk_overlap):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

def chunk_documents(documents, chunk_size, chunk_overlap):
    splitter = make_splitter(chunk_size, chunk_overlap)
    chunks = splitter.split_documents(documents)

    for i, doc in enumerate(chunks):
        doc.metadata["chunk_id"] = f"chunk_{i}"
        doc.metadata["chunk_index"] = i

    return chunks

def normalized_tokens(text):
    return re.findall(r"\w+", text.lower())

def answer_in_chunk(chunk_text, answers):
    normalized_chunk = " ".join(normalized_tokens(chunk_text))
    for ans in answers:
        normalized_ans = " ".join(normalized_tokens(ans))
        if normalized_ans and normalized_ans in normalized_chunk:
            return True
    return False

# Use the full unique training corpus for chunking,
# but only development questions for measuring configuration quality.
tuning_rows = []

for chunk_size, overlap in CHUNK_CONFIGS:
    chunks = chunk_documents(train_documents, chunk_size, overlap)

    tokenized = [normalized_tokens(d.page_content) for d in chunks]
    bm25_dev = BM25Okapi(tokenized)

    context_to_chunks = {}
    for idx, d in enumerate(chunks):
        context_to_chunks.setdefault(d.metadata["context_id"], []).append(idx)

    hits_at_5 = 0
    hits_at_10 = 0

    for ex in dev_tuning_examples:
        key = context_key(ex["context"])
        context_id = hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]

        scores = bm25_dev.get_scores(normalized_tokens(ex["question"]))
        ranked = np.argsort(scores)[::-1]

        def hit_at(k):
            for idx in ranked[:k]:
                d = chunks[int(idx)]
                if d.metadata["context_id"] == context_id and answer_in_chunk(d.page_content, ex["answers"]):
                    return True
            return False

        hits_at_5 += int(hit_at(5))
        hits_at_10 += int(hit_at(10))

    tuning_rows.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "num_chunks": len(chunks),
        "dev_recall@5": hits_at_5 / len(dev_tuning_examples),
        "dev_recall@10": hits_at_10 / len(dev_tuning_examples)
    })

tuning_df = pd.DataFrame(tuning_rows)
display(tuning_df)

# Selection rule: highest Recall@10, then fewer chunks.
best_row = tuning_df.sort_values(
    ["dev_recall@10", "num_chunks"],
    ascending=[False, True]
).iloc[0]

CHUNK_SIZE = int(best_row["chunk_size"])
CHUNK_OVERLAP = int(best_row["overlap"])

print(f"Selected chunk size: {CHUNK_SIZE}")
print(f"Selected overlap: {CHUNK_OVERLAP}")

,chunk_size,overlap,num_chunks,dev_recall@5,dev_recall@10
0,800,100,25733,0.72,0.79
1,1200,200,20410,0.76,0.79
2,1600,300,19213,0.76,0.83


Selected chunk size: 1600
Selected overlap: 300


## 9. Build the final validation knowledge base

The final retrieval corpus contains **only unique validation contexts**.

This is intentional: the validation questions must have their corresponding passages available in the RAG knowledge base so that Recall@K measures retrieval quality rather than accidentally measuring whether the train corpus contains a validation article.

In [ ]:
validation_chunks = chunk_documents(
    validation_documents,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

In [ ]:
validation_chunks = chunk_documents(
    validation_documents,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

all_chunks = validation_chunks
all_chunk_texts = [d.page_content for d in all_chunks]
all_chunk_ids = [d.metadata["chunk_id"] for d in all_chunks]

print("Final validation chunks:", len(all_chunks))
print("First metadata:", all_chunks[0].metadata)

Final validation chunks: 2127
First metadata: {'context_id': 'c64554d86b5e183d', 'title': 'Super_Bowl_50', 'chunk_id': 'chunk_0', 'chunk_index': 0}


## 10. BM25 sparse retriever

In [ ]:
tokenized_chunks = [normalized_tokens(text) for text in all_chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_retrieve(query, top_k=50):
    scores = bm25.get_scores(normalized_tokens(query))
    ranked = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "chunk_idx": int(idx),
            "chunk_id": all_chunks[int(idx)].metadata["chunk_id"],
            "text": all_chunks[int(idx)].page_content,
            "score": float(scores[int(idx)])
        }
        for idx in ranked
    ]

print("BM25 ready.")

BM25 ready.


## 11. Sentence-Transformers dense embeddings

In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)

chunk_embeddings = embedding_model.encode(
    all_chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Embedding shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Embedding shape: (2127, 384)


## 12. ChromaDB dense vector store through LangChain

Every document gets an explicit `chunk_id` in metadata. We do not search through all chunks to rediscover which chunk Chroma returned.

In [ ]:
CHROMA_DIR = "/content/chroma_squad_rag"
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

embedding_function = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

chroma_db = Chroma(
    collection_name="squad_validation_chunks",
    embedding_function=embedding_function,
    persist_directory=CHROMA_DIR
)

chroma_db.add_documents(
    documents=all_chunks,
    ids=all_chunk_ids
)

print("ChromaDB documents:", chroma_db._collection.count())
print("Persist directory:", CHROMA_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB documents: 2127
Persist directory: /content/chroma_squad_rag


## 13. Dense retrieval

In [ ]:
def dense_retrieve(query, top_k=50):
    results = chroma_db.similarity_search_with_score(query, k=top_k)

    output = []

    for doc, distance in results:
        output.append({
            "chunk_id": doc.metadata["chunk_id"],
            "chunk_idx": int(doc.metadata["chunk_index"]),
            "text": doc.page_content,
            "distance": float(distance)
        })

    return output

print(dense_retrieve("To what two islands does the ferry service connect?", top_k=3)[0]["text"][:500])

From Newcastle International Ferry Terminal, at North Shields, Danish DFDS Seaways run a service to IJmuiden (near Amsterdam). The DFDS ferry service to Gothenburg, Sweden, ceased at the end of October 2006 – the company cited high fuel prices and new competition from low-cost air services as the cause – and their service to Bergen and Stavanger, Norway was terminated late 2008. Since summer 2007, Thomson cruise lines have included Newcastle as a departure port on its Norwegian and Fjords cruise


## 14. Reciprocal Rank Fusion (RRF)

RRF combines the **rank positions** from BM25 and Chroma dense retrieval.

Formula:

**RRF(d) = Σ 1 / (k + rank(d))**

The project uses BM25 + dense retrieval first, then a cross-encoder reranker.

In [ ]:
RRF_K = 60
CANDIDATE_K = 50
RERANK_TOP_K = 3

def reciprocal_rank_fusion(bm25_results, dense_results, top_k=50, rrf_k=60):
    scores = {}
    lookup = {}

    for rank, item in enumerate(bm25_results, start=1):
        cid = item["chunk_id"]
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (rrf_k + rank)
        lookup[cid] = item

    for rank, item in enumerate(dense_results, start=1):
        cid = item["chunk_id"]
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (rrf_k + rank)
        lookup[cid] = item

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    fused = []
    for cid, score in ranked[:top_k]:
        item = lookup[cid].copy()
        item["rrf_score"] = float(score)
        fused.append(item)

    return fused

def hybrid_retrieve(query, candidate_k=CANDIDATE_K):
    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    dense_results = dense_retrieve(query, top_k=candidate_k)

    return reciprocal_rank_fusion(
        bm25_results,
        dense_results,
        top_k=candidate_k,
        rrf_k=RRF_K
    )

print("Hybrid retrieval ready.")

Hybrid retrieval ready.


## 15. Cross-Encoder reranker

In [ ]:
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    RERANKER_NAME,
    device=DEVICE
)

def rerank_results(query, candidates, top_k=RERANK_TOP_K):
    if not candidates:
        return []

    pairs = [(query, item["text"]) for item in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: float(x[1]),
        reverse=True
    )

    output = []
    for item, score in ranked[:top_k]:
        new_item = item.copy()
        new_item["rerank_score"] = float(score)
        output.append(new_item)

    return output

def retrieve(query, candidate_k=CANDIDATE_K, final_k=RERANK_TOP_K):
    candidates = hybrid_retrieve(query, candidate_k=candidate_k)
    return rerank_results(query, candidates, top_k=final_k)

print("Reranker ready.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker ready.


## 16. Retrieval sanity check

In [ ]:
test_question = "To what two islands does the ferry service connect?"

results = retrieve(test_question, candidate_k=50, final_k=3)

for i, item in enumerate(results, 1):
    print(f"\nRank {i}")
    print("Chunk ID:", item["chunk_id"])
    print("RRF:", item["rrf_score"])
    print("Reranker:", item["rerank_score"])
    print(item["text"][:1000])


Rank 1
Chunk ID: chunk_1018
RRF: 0.03278688524590164
Reranker: -4.436863422393799
From Newcastle International Ferry Terminal, at North Shields, Danish DFDS Seaways run a service to IJmuiden (near Amsterdam). The DFDS ferry service to Gothenburg, Sweden, ceased at the end of October 2006 – the company cited high fuel prices and new competition from low-cost air services as the cause – and their service to Bergen and Stavanger, Norway was terminated late 2008. Since summer 2007, Thomson cruise lines have included Newcastle as a departure port on its Norwegian and Fjords cruise.

Rank 2
Chunk ID: chunk_1653
RRF: 0.015384615384615385
Reranker: -10.333967208862305
The Swahili built Mombasa into a major port city and established trade links with other nearby city-states, as well as commercial centres in Persia, Arabia, and even India. By the 15th-century, Portuguese voyager Duarte Barbosa claimed that "Mombasa is a place of great traffic and has a good harbour in which there are always moo

## 17. Prepare validation evaluation set

Default is 50 questions for a practical Colab run. Set `FULL_VALIDATION_EVAL = True` if you want the complete SQuAD validation evaluation; this can take substantially longer because every question invokes retrieval, reranking and TinyLlama.

In [ ]:
FULL_VALIDATION_EVAL = False
N_EVAL = 50

eval_rng = random.Random(SEED)

if FULL_VALIDATION_EVAL:
    validation_examples = validation_examples_all
else:
    validation_examples = eval_rng.sample(
        validation_examples_all,
        min(N_EVAL, len(validation_examples_all))
    )

print("Evaluation questions:", len(validation_examples))

Evaluation questions: 50


## 18. Retrieval evaluation

A retrieval hit requires:

1. the retrieved chunk belongs to the question's gold context, and
2. the retrieved chunk contains a normalized form of one of the gold answer strings.

This is stricter than merely finding a chunk from the same article.

In [ ]:
def gold_context_id(example):
    key = context_key(example["context"])
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]

def evaluate_retrieval(examples, ks=(1, 3, 5, 10), candidate_k=50):
    hits = {k: 0 for k in ks}
    reciprocal_ranks = []
    rows = []

    for n, ex in enumerate(examples, 1):
        results = retrieve(
            ex["question"],
            candidate_k=candidate_k,
            final_k=max(ks)
        )

        gold_cid = gold_context_id(ex)
        first_hit_rank = None

        for rank, item in enumerate(results, 1):
            same_context = item["chunk_id"] in {
                d.metadata["chunk_id"]
                for d in validation_chunks
                if d.metadata["context_id"] == gold_cid
            }

            has_answer = answer_in_chunk(item["text"], ex["answers"])

            if same_context and has_answer:
                first_hit_rank = rank
                break

        for k in ks:
            if first_hit_rank is not None and first_hit_rank <= k:
                hits[k] += 1

        reciprocal_ranks.append(
            1.0 / first_hit_rank if first_hit_rank is not None else 0.0
        )

        rows.append({
            "id": ex["id"],
            "question": ex["question"],
            "first_hit_rank": first_hit_rank
        })

        if n % 10 == 0 or n == len(examples):
            print(f"{n}/{len(examples)}")

    metrics = {
        f"Recall@{k}": hits[k] / len(examples)
        for k in ks
    }
    metrics["MRR"] = float(np.mean(reciprocal_ranks))

    return metrics, pd.DataFrame(rows)

retrieval_metrics, retrieval_rows = evaluate_retrieval(
    validation_examples,
    ks=(1, 3, 5, 10),
    candidate_k=CANDIDATE_K
)

print("\nRetrieval results")
for k, v in retrieval_metrics.items():
    print(f"{k}: {v:.4f}")

display(retrieval_rows.head())

10/50
20/50
30/50
40/50
50/50

Retrieval results
Recall@1: 0.8800
Recall@3: 0.9800
Recall@5: 0.9800
Recall@10: 1.0000
MRR: 0.9322


,id,question,first_hit_rank
0,573786b51c4567190057448d,How might gravity effects be observed differen...,1
1,56e1ddfce3433e14004231d8,What is the prize offered for finding a soluti...,1
2,56bf48cc3aeaaa14008c95af,What color were the Bronco's uniforms in Super...,1
3,5725c337271a42140099d164,What are the little tentacles that cydippids h...,2
4,5725e44238643c19005ace36,Which lunar probe was near the Apollo 12 crew'...,1


## 19. TinyLlama 1.1B Chat model

This is the main answer generator in the project. The retriever/reranker supplies evidence; TinyLlama generates the final answer from that evidence.

The prompt explicitly asks for a short answer supported only by the supplied context.

In [ ]:
LLM_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)

if DEVICE == "cpu":
    llm_model = llm_model.to(DEVICE)

if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print("TinyLlama loaded.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded.


## 20. End-to-end RAG answer function

In [ ]:
def build_prompt(question, contexts):
    context_text = "\n\n".join(
        [f"[Context {i+1}]\n{c}" for i, c in enumerate(contexts)]
    )

    return f"""<|system|>
You are a question-answering assistant.
Answer the question using only the supplied context.
Give a concise answer.
If the answer is not present in the context, say: Not found in context.
Do not add unrelated explanation.
<|user|>
Context:
{context_text}

Question:
{question}
<|assistant|>
"""

@torch.inference_mode()
def generate_tinyllama(prompt, max_new_tokens=64):
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    output = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()

def rag_answer(question, candidate_k=50, final_k=3):
    reranked = retrieve(
        question,
        candidate_k=candidate_k,
        final_k=final_k
    )

    contexts = [item["text"] for item in reranked]

    prompt = build_prompt(question, contexts)
    answer = generate_tinyllama(prompt)

    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "retrieved": reranked
    }

demo = rag_answer(
    "To what two islands does the ferry service connect?",
    candidate_k=50,
    final_k=3
)

print("Answer:", demo["answer"])
print("\nRetrieved context:")
for i, c in enumerate(demo["contexts"], 1):
    print(f"\n--- Context {i} ---\n{c[:1000]}")

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: The ferry service connects Newcastle International Ferry Terminal to IJmuiden (near Amsterdam) and to Bergen and Stavanger, Norway.

Retrieved context:

--- Context 1 ---
From Newcastle International Ferry Terminal, at North Shields, Danish DFDS Seaways run a service to IJmuiden (near Amsterdam). The DFDS ferry service to Gothenburg, Sweden, ceased at the end of October 2006 – the company cited high fuel prices and new competition from low-cost air services as the cause – and their service to Bergen and Stavanger, Norway was terminated late 2008. Since summer 2007, Thomson cruise lines have included Newcastle as a departure port on its Norwegian and Fjords cruise.

--- Context 2 ---
The Swahili built Mombasa into a major port city and established trade links with other nearby city-states, as well as commercial centres in Persia, Arabia, and even India. By the 15th-century, Portuguese voyager Duarte Barbosa claimed that "Mombasa is a place of great traffic and has a good harbour

## 21. Token-level answer metrics

For each question, multiple SQuAD reference answers are allowed. The implementation takes the best matching reference for the prediction.

These are **token-level Precision, Recall and F1**, not BERTScore.

In [ ]:
def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text

def token_prf(prediction, reference):
    pred = normalize_answer(prediction).split()
    ref = normalize_answer(reference).split()

    if not pred and not ref:
        return 1.0, 1.0, 1.0
    if not pred or not ref:
        return 0.0, 0.0, 0.0

    common = {}
    for token in pred:
        common[token] = common.get(token, 0) + 1

    overlap = 0
    ref_counts = {}
    for token in ref:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    for token in common:
        overlap += min(common[token], ref_counts.get(token, 0))

    precision = overlap / len(pred)
    recall = overlap / len(ref)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

def best_reference_metrics(prediction, references):
    scores = [
        token_prf(prediction, ref)
        for ref in references
    ]
    return max(scores, key=lambda x: x[2])

def exact_match(prediction, references):
    pred = normalize_answer(prediction)
    return int(any(pred == normalize_answer(ref) for ref in references))

## 22. End-to-end answer evaluation

In [ ]:
def evaluate_answers(examples):
    rows = []

    for n, ex in enumerate(examples, 1):
        result = rag_answer(
            ex["question"],
            candidate_k=CANDIDATE_K,
            final_k=RERANK_TOP_K
        )

        pred = result["answer"]
        precision, recall, f1 = best_reference_metrics(
            pred,
            ex["answers"]
        )

        rows.append({
            "id": ex["id"],
            "question": ex["question"],
            "prediction": pred,
            "references": ex["answers"],
            "EM": exact_match(pred, ex["answers"]),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "retrieved_contexts": result["contexts"]
        })

        if n % 5 == 0 or n == len(examples):
            print(f"{n}/{len(examples)}")

    return pd.DataFrame(rows)

answer_results = evaluate_answers(validation_examples)

answer_metrics = {
    "Exact Match": answer_results["EM"].mean(),
    "Token Precision": answer_results["Precision"].mean(),
    "Token Recall": answer_results["Recall"].mean(),
    "Token F1": answer_results["F1"].mean()
}

print("\nAnswer results")
for k, v in answer_metrics.items():
    print(f"{k}: {v:.4f}")

display(answer_results[[
    "question", "prediction", "references", "EM",
    "Precision", "Recall", "F1"
]].head(10))

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

5/50


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

10/50


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

...

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

45/50


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

50/50

Answer results
Exact Match: 0.0000
Token Precision: 0.1282
Token Recall: 0.7292
Token F1: 0.2041


,question,prediction,references,EM,Precision,Recall,F1
0,How might gravity effects be observed differen...,"According to Newton, the effects of gravity mi...","[at larger distances., at larger distances, at...",0,0.062500,1.000,0.117647
1,What is the prize offered for finding a soluti...,The prize offered for finding a solution to P=...,"[$1,000,000, US$1,000,000, US$1,000,000]",0,0.066667,1.000,0.125000
2,What color were the Bronco's uniforms in Super...,The Broncos' uniforms in Super Bowl 50 were wh...,"[white, matching white jerseys, white]",0,0.125000,1.000,0.222222
3,What are the little tentacles that cydippids h...,The little tentacles of cydippids are called t...,"[tentilla, tentilla, tentilla]",0,0.142857,1.000,0.250000
4,Which lunar probe was near the Apollo 12 crew'...,The lunar probe near the Apollo 12 crew's land...,"[Surveyor 3, Surveyor 3, Surveyor 3, Surveyor ...",0,0.181818,1.000,0.307692
5,Electrolysis of what can be used to produce ox...,Electrolysis of water into molecular oxygen an...,"[water, water, water, water]",0,0.125000,1.000,0.222222
6,When was the Edict of Worms presented?,The Edict of Worms presented on 25 May 1521.,"[25 May 1521, 25 May 1521, 25 May 1521]",0,0.375000,1.000,0.545455
7,How many other important people sent letters?,There are at least 50 letters from other impor...,"[more than 70, more than 70, more than 70]",0,0.000000,0.000,0.000000
8,What is an alternate way to make starch?,An alternate way to make starch is through the...,[glucose monomers in the chloroplast can be li...,0,0.078947,0.375,0.130435
9,What type of image was Tesla thought to have a...,Tesla may have inadvertently captured an X-ray...,"[X-ray image, X-ray, X-ray image]",0,0.083333,1.000,0.153846


## 23. BERTScore

BERTScore is calculated against **all available reference answers**, and the best F1 reference is selected for each prediction.

This is separate from the token-level Precision / Recall / F1 above.

In [ ]:
from bert_score import score as bert_score

predictions = answer_results["prediction"].tolist()
reference_lists = answer_results["references"].tolist()

# BERTScore expects one reference per prediction in this simple evaluation.
# We therefore evaluate every prediction against all references and keep
# the reference producing the highest F1.
all_p = []
all_r = []
all_f = []

for pred, refs in zip(predictions, reference_lists):
    p, r, f = bert_score(
        [pred] * len(refs),
        refs,
        lang="en",
        verbose=False
    )

    best_idx = int(torch.argmax(f).item())
    all_p.append(float(p[best_idx]))
    all_r.append(float(r[best_idx]))
    all_f.append(float(f[best_idx]))

bert_metrics = {
    "BERTScore Precision": float(np.mean(all_p)),
    "BERTScore Recall": float(np.mean(all_r)),
    "BERTScore F1": float(np.mean(all_f))
}

print("\nBERTScore results")
for k, v in bert_metrics.items():
    print(f"{k}: {v:.4f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERTScore results
BERTScore Precision: 0.8301
BERTScore Recall: 0.8834
BERTScore F1: 0.8557


## 24. Error analysis

This identifies examples where the answer was not an exact match or where token F1 was low.

In [ ]:
error_analysis = answer_results.sort_values(
    ["F1", "EM"],
    ascending=[True, True]
)

display(error_analysis[[
    "question",
    "prediction",
    "references",
    "EM",
    "Precision",
    "Recall",
    "F1"
]].head(10))

,question,prediction,references,EM,Precision,Recall,F1
7,How many other important people sent letters?,There are at least 50 letters from other impor...,"[more than 70, more than 70, more than 70]",0,0.0,0.0,0.0
22,In what process are electrons recycled?,"In the context of photosystem II, electrons ar...","[cyclic photophosphorylation, cyclic photophos...",0,0.0,0.0,0.0
24,What objects in organisms absorb singlet oxyge...,The answer to the question is: Reactive oxygen...,"[Carotenoids, Carotenoids, Carotenoids, Carote...",0,0.0,0.0,0.0
26,"What interpretation of Islam is, for many of t...",The interpretation of Islam promoted by this f...,"[Saudi, the Saudi-interpretation, Saudi]",0,0.0,0.0,0.0
32,What does ctenophore use to capture prey?,The ctenophore uses its tentacles and tentilla...,"[colloblasts, colloblasts, colloblasts]",0,0.0,0.0,0.0
36,How many types of science fiction have been im...,There are at least 278 patents issued to Tesla...,"[several, several, several]",0,0.0,0.0,0.0
37,What did he do with Wardenclyffe?,Answer: Tesla did not complete the constructio...,"[sold, sold, sold]",0,0.0,0.0,0.0
38,Whose plan called for the Mongolian army to sp...,The plan called for the Mongolian army to spli...,"[Subutai, Subutai, Subutai]",0,0.0,0.0,0.0
40,When did Sunday Night Football premiere on NBC?,Sunday Night Football premiered on NBC on Sept...,"[2006, 2006, 2006]",0,0.0,0.0,0.0
42,Who was made rich and prosperous prior to Worl...,The answer to the question is:\n\nHistorians J...,"[many imperial powers, imperial powers, imperi...",0,0.0,0.0,0.0


## 25. Optional extractive QA comparison

This is an **additional experiment**, not the main project generator.

It can be used to compare TinyLlama's generative answers with a RoBERTa SQuAD2-style extractive QA model over the retrieved contexts.

In [ ]:
RUN_ROBERTA_COMPARISON = False

if RUN_ROBERTA_COMPARISON:
    from transformers import AutoTokenizer as QA_AutoTokenizer
    from transformers import AutoModelForQuestionAnswering

    QA_MODEL_NAME = "deepset/roberta-base-squad2"

    qa_tokenizer = QA_AutoTokenizer.from_pretrained(QA_MODEL_NAME)
    qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME).to(DEVICE)
    qa_model.eval()

    print("RoBERTa extractive QA model loaded.")
else:
    print("RoBERTa comparison disabled. Set RUN_ROBERTA_COMPARISON=True to run it.")

RoBERTa comparison disabled. Set RUN_ROBERTA_COMPARISON=True to run it.


## 26. ChromaDB persistence / reload check

The vector store is persisted at `/content/chroma_squad_rag`.

This check verifies that the stored Chroma collection can be opened again with the same embedding function.

In [ ]:
reloaded_chroma = Chroma(
    collection_name="squad_validation_chunks",
    embedding_function=embedding_function,
    persist_directory=CHROMA_DIR
)

print("Reloaded Chroma count:", reloaded_chroma._collection.count())

reload_test = reloaded_chroma.similarity_search(
    "To what two islands does the ferry service connect?",
    k=1
)

print("Reload successful:", len(reload_test) == 1)

Reloaded Chroma count: 2127
Reload successful: True


# 27. Final project summary

### Implemented components

- SQuAD 1.1 dataset
- Data cleaning
- Duplicate context removal
- LangChain `Document`
- LangChain `RecursiveCharacterTextSplitter`
- Chunk-size development tuning
- Sentence-Transformers `all-MiniLM-L6-v2`
- ChromaDB dense retrieval
- BM25 sparse retrieval
- Reciprocal Rank Fusion
- Candidate Top-50 retrieval
- Cross-Encoder reranking
- Final Top-3 evidence
- TinyLlama-1.1B-Chat answer generation
- Exact Match
- Token Precision / Recall / F1
- Multi-reference BERTScore Precision / Recall / F1
- Retrieval Recall@1 / @3 / @5 / @10
- MRR
- Error analysis
- Reproducibility seed
- ChromaDB persistence/reload
- End-to-end RAG function

### Final architecture

SQuAD
↓
Data Cleaning
↓
Duplicate Context Removal
↓
LangChain Documents
↓
LangChain Chunking
↓
┌───────────────┐       ┌──────────────────┐
│ BM25          │       │ ChromaDB         │
│ Sparse        │       │ Dense Retrieval  │
└──────┬────────┘       └────────┬─────────┘
       └──────────────┬──────────┘
                      ↓
               Reciprocal Rank Fusion
                      ↓
                 Candidate Top-50
                      ↓
             Cross-Encoder Reranker
                      ↓
                    Top-3
                      ↓
                  TinyLlama
                1.1B Chat Model
                      ↓
                   Answer
                      ↓
       Precision / Recall / F1 / BERTScore

## 28. Results table for your report/resume notes

Run this final cell after the evaluations finish.

In [ ]:
final_results = {
    **retrieval_metrics,
    **answer_metrics,
    **bert_metrics,
    "Chunk size": CHUNK_SIZE,
    "Chunk overlap": CHUNK_OVERLAP,
    "Candidate K": CANDIDATE_K,
    "Final reranked K": RERANK_TOP_K,
    "Evaluation questions": len(validation_examples)
}

results_df = pd.DataFrame(
    list(final_results.items()),
    columns=["Metric", "Value"]
)

display(results_df)

,Metric,Value
0,Recall@1,0.880000
1,Recall@3,0.980000
2,Recall@5,0.980000
3,Recall@10,1.000000
4,MRR,0.932222
5,Exact Match,0.000000
6,Token Precision,0.128194
7,Token Recall,0.729167
8,Token F1,0.204142
9,BERTScore Precision,0.830119
